In [4]:
print('Helo')

Helo


In [3]:
import pandas as pd

train_transaction = pd.read_csv("../data/raw/train_transaction.csv")
train_identity = pd.read_csv("../data/raw/train_identity.csv")

print("Transaction dataset shape:", train_transaction.shape)
print("Identity dataset shape:", train_identity.shape)

Transaction dataset shape: (590540, 394)
Identity dataset shape: (144233, 41)


In [5]:
train_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
print("Data types:")
print(train_transaction.dtypes.value_counts())

print("\nMissing values — top 20:")
print(train_transaction.isna().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
print(train_transaction["isFraud"].value_counts())

print("\nTarget percentage:")
print(train_transaction["isFraud"].value_counts(normalize=True) * 100)

Data types:
float64    376
object      14
int64        4
Name: count, dtype: int64

Missing values — top 20:
dist2    552913
D7       551623
D13      528588
D14      528353
D12      525823
D6       517353
D9       515614
D8       515614
V153     508595
V139     508595
V162     508595
V161     508595
V154     508595
V138     508595
V158     508595
V157     508595
V163     508595
V156     508595
V155     508595
V149     508595
dtype: int64

Target distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Target percentage:
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


## Initial Dataset Observations (Transactions.csv)

### Dataset Structure
- The transaction dataset contains **590,540 transactions across 394 features**.
- The identity dataset contains **144,233 records across 41 features**.
- Since the number of identity records is substantially lower than the number of transactions, **identity information is not available for every transaction**.

### Data Quality
- A large number of variables contain substantial missing values.
- Some variables have more than **500,000 missing values**, indicating that missingness is a fundamental characteristic of this dataset rather than an isolated data-quality issue.
- We should investigate whether **missingness itself is associated with fraud** before deciding how to handle these variables.

### Target Distribution
- Only **3.50% of transactions are fraudulent**, while 96.50% are legitimate.
- This represents a significant **class imbalance**.
- A model that predicts every transaction as legitimate could achieve approximately 96.5% accuracy while detecting no fraud. Therefore, accuracy alone would be a misleading evaluation metric.
- The modeling stage should prioritize **Precision, Recall, PR-AUC and threshold-based business metrics**.

### Initial Modeling Implications
- The dataset is large and highly dimensional, so using all features blindly may introduce unnecessary noise, computational cost, and redundant information.
- Feature selection should therefore be based on **data quality, availability at prediction time, leakage risk, redundancy, and predictive contribution**.
- The high level of missingness also makes **missing-value patterns potentially useful features** rather than something that should automatically be discarded.

In [9]:
columns = train_transaction.columns.tolist()

for prefix in ["Transaction", "Product", "card", "addr", "dist",
               "C", "D", "M", "V", "P_email", "R_email"]:
    
    matching = [col for col in columns if col.startswith(prefix)]
    print(f"{prefix:10} : {len(matching)} columns")

print("\nTime-related columns:")
print([col for col in columns if "DT" in col or "time" in col.lower()])

Transaction : 3 columns
Product    : 1 columns
card       : 6 columns
addr       : 2 columns
dist       : 2 columns
C          : 14 columns
D          : 15 columns
M          : 9 columns
V          : 339 columns
P_email    : 1 columns
R_email    : 1 columns

Time-related columns:
['TransactionDT']


## Feature Family Observations

- The transaction dataset contains several distinct feature families, including transaction, product, card, address, distance, email, C, D, M, and V features.
- `TransactionDT` is the primary time-related variable and represents the relative timing of transactions.
- The dataset is heavily dominated by **V-features, with 339 of the 394 columns belonging to this family**.
- Since V-features are anonymized, their business meaning cannot be reliably inferred from their names alone.
- Therefore, feature selection will focus on **data quality, missingness, predictive contribution, leakage risk, and temporal stability** rather than assigning unsupported business interpretations to anonymized variables.
- The interpretable feature groups such as transaction amount, product, card, address, email, and time will be particularly useful for explaining fraud patterns.

In [10]:
print(train_transaction[[
    "TransactionID",
    "TransactionDT",
    "TransactionAmt",
    "isFraud"
]].head())

   TransactionID  TransactionDT  TransactionAmt  isFraud
0        2987000          86400            68.5        0
1        2987001          86401            29.0        0
2        2987002          86469            59.0        0
3        2987003          86499            50.0        0
4        2987004          86506            50.0        0


In [11]:
print("TransactionID unique values:", train_transaction["TransactionID"].nunique())

print("\nTransactionDT range:")
print(train_transaction["TransactionDT"].min(), "to", train_transaction["TransactionDT"].max())

print("\nTransactionAmt summary:")
print(train_transaction["TransactionAmt"].describe())


TransactionID unique values: 590540

TransactionDT range:
86400 to 15811131

TransactionAmt summary:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64


### Core Transaction Variable Observations

- `TransactionID` contains **590,540 unique values for 590,540 transactions**, confirming that each row represents a unique transaction.

- `TransactionID` will be retained for transaction identification and dataset joins but excluded as a modeling feature because it is an identifier rather than a meaningful behavioral variable.

- `TransactionDT` ranges from **86,400 to 15,811,131** and represents relative transaction timing. The approximately six-month temporal span makes time-aware validation particularly relevant.

- `TransactionAmt` has a median of approximately **68.77** while the mean is approximately **135.03**, suggesting a right-skewed transaction amount distribution caused by relatively large transactions.

- Transaction amount will therefore be investigated further rather than assuming that higher-value transactions are inherently more fraudulent.

- `TransactionDT` and `TransactionAmt` will be retained for feature engineering and fraud-pattern analysis.

In [12]:
print("Transaction rows:", len(train_transaction))
print("Identity rows:", len(train_identity))

print("\nUnique TransactionIDs in identity:")
print(train_identity["TransactionID"].nunique())

print("\nIdentity coverage of transactions:")
print(
    train_identity["TransactionID"].isin(
        train_transaction["TransactionID"]
    ).mean() * 100
)

Transaction rows: 590540
Identity rows: 144233

Unique TransactionIDs in identity:
144233

Identity coverage of transactions:
100.0


In [13]:
identity_coverage = (
    train_transaction["TransactionID"].isin(
        train_identity["TransactionID"]
    ).mean() * 100
)

print(f"Transactions with identity data: {identity_coverage:.2f}%")
print(f"Transactions without identity data: {100 - identity_coverage:.2f}%")

Transactions with identity data: 24.42%
Transactions without identity data: 75.58%


In [14]:
train_transaction["has_identity"] = train_transaction["TransactionID"].isin(
    train_identity["TransactionID"]
)

print(
    train_transaction.groupby("has_identity")["isFraud"]
    .agg(["count", "sum", "mean"])
)

               count    sum      mean
has_identity                         
False         446307   9345  0.020939
True          144233  11318  0.078470


## Identity Availability and Fraud

- Identity information is available for **144,233 out of 590,540 transactions**, corresponding to approximately **24.42%** of all transactions.

- Transactions with identity information have a fraud rate of **7.85%**, compared with **2.09%** for transactions without identity information.

- Therefore, transactions with an associated identity record show a substantially higher observed fraud rate in this dataset.

- This indicates that **identity-data availability is associated with the fraud outcome** and may contain predictive information.

- The presence or absence of identity information will therefore be considered as a potential **missingness/availability feature** during feature engineering.

- This relationship should be treated as an **association rather than a causal relationship**, since the analysis does not establish that identity availability causes fraud.

# Inspect the identity table

In [15]:
print("Identity columns:")
print(train_identity.columns.tolist())

print("\nData types:")
print(train_identity.dtypes.value_counts())

print("\nTop 20 missing-value columns:")
print(train_identity.isna().sum().sort_values(ascending=False).head(20))

Identity columns:
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

Data types:
float64    23
object     17
int64       1
Name: count, dtype: int64

Top 20 missing-value columns:
id_24         139486
id_25         139101
id_07         139078
id_08         139078
id_21         139074
id_26         139070
id_23         139064
id_27         139064
id_22         139064
id_18          99120
id_03          77909
id_04          77909
id_33          70944
id_09          69307
id_10          69307
id_30          66668
id_32          66647
id_34          66428
id_14          64189
DeviceInfo     25567
dtype: int64


## Identity Table Observations

- The identity dataset contains **41 columns**, consisting of `TransactionID`, 38 anonymized `id_*` features, `DeviceType`, and `DeviceInfo`.

- The identity features contain a mixture of numerical and categorical variables, with **23 float, 17 object, and 1 integer column**.

- Missingness is substantial across several identity variables. For example, `id_24` is missing for **139,486 of 144,233 identity records**, representing approximately 96.7% missingness.

- `DeviceInfo` has comparatively lower missingness, with **25,567 missing values**, and may therefore be more usable for downstream analysis.

- The anonymized `id_*` variables will not be assigned business meanings without reliable documentation.

- High missingness alone will not be used as an automatic reason to remove a feature. Feature selection will consider **missingness, predictive contribution, leakage risk, temporal stability, redundancy, and availability at prediction time**.

In [26]:
print(train_identity[[
    "TransactionID",
    "DeviceType",
    "DeviceInfo"
]].head())

print("\nUnique DeviceType values:")
print(train_identity["DeviceType"].value_counts(dropna=False))

print("\nTop DeviceInfo values:")
print(train_identity["DeviceInfo"].value_counts(dropna=False).head(15))

   TransactionID DeviceType                     DeviceInfo
0        2987004     mobile  SAMSUNG SM-G892A Build/NRD90M
1        2987008     mobile                     iOS Device
2        2987010    desktop                        Windows
3        2987011    desktop                            NaN
4        2987016    desktop                          MacOS

Unique DeviceType values:
DeviceType
desktop    85165
mobile     55645
NaN         3423
Name: count, dtype: int64

Top DeviceInfo values:
DeviceInfo
Windows                        47722
NaN                            25567
iOS Device                     19782
MacOS                          12573
Trident/7.0                     7440
rv:11.0                         1901
rv:57.0                          962
SM-J700M Build/MMB29K            549
SM-G610M Build/MMB29K            461
SM-G531H Build/LMY48B            410
rv:59.0                          362
SM-G935F Build/NRD90M            334
SM-G955U Build/NRD90M            328
SM-G532M Build/

In [32]:
print("Total Unique Devices:", train_identity['DeviceInfo'].unique().shape[0])

Total Unique Devices: 1787


In [34]:
df_with_device = train_transaction.merge(
    train_identity[["TransactionID", "DeviceType"]],
    on="TransactionID",
    how="left"
)

device_fraud = df_with_device.groupby("DeviceType")["isFraud"].agg(
    transaction_count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

print("=== Fraud Rate by Device Type ===\n")

for device, row in device_fraud.iterrows():
    print(f"Device Type       : {device}")
    print(f"Transactions      : {row['transaction_count']:,}")
    print(f"Fraud Cases       : {row['fraud_count']:,}")
    print(f"Fraud Rate        : {row['fraud_rate'] * 100:.2f}%")
    print("-" * 40)

=== Fraud Rate by Device Type ===

Device Type       : desktop
Transactions      : 85,165.0
Fraud Cases       : 5,554.0
Fraud Rate        : 6.52%
----------------------------------------
Device Type       : mobile
Transactions      : 55,645.0
Fraud Cases       : 5,657.0
Fraud Rate        : 10.17%
----------------------------------------


In [35]:
df_with_device["DeviceType"] = df_with_device["DeviceType"].fillna("Missing")

device_fraud = df_with_device.groupby("DeviceType")["isFraud"].agg(
    transaction_count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

print("=== Fraud Rate by Device Type ===\n")

for device, row in device_fraud.iterrows():
    print(f"Device Type       : {device}")
    print(f"Transactions      : {row['transaction_count']:,}")
    print(f"Fraud Cases       : {row['fraud_count']:,}")
    print(f"Fraud Rate        : {row['fraud_rate'] * 100:.2f}%")
    print("-" * 40)

=== Fraud Rate by Device Type ===

Device Type       : Missing
Transactions      : 449,730.0
Fraud Cases       : 9,452.0
Fraud Rate        : 2.10%
----------------------------------------
Device Type       : desktop
Transactions      : 85,165.0
Fraud Cases       : 5,554.0
Fraud Rate        : 6.52%
----------------------------------------
Device Type       : mobile
Transactions      : 55,645.0
Fraud Cases       : 5,657.0
Fraud Rate        : 10.17%
----------------------------------------


### Device Type and Fraud

- When considering all transactions, `DeviceType` is available as desktop for **85,165 transactions** and mobile for **55,645 transactions**. It is missing for **449,730 transactions**.

- The observed fraud rate is **6.52% for desktop**, **10.17% for mobile**, and **2.10% for transactions with missing device type**.

- Among transactions with a recorded device type, mobile transactions have a higher observed fraud rate than desktop transactions.

- The missing `DeviceType` category should not be interpreted simply as a missing value within the identity table, because it also captures transactions for which no identity record is available.

- Device type and identity availability therefore provide related but potentially distinct information.

- `has_identity` and `DeviceType` will both be considered during feature engineering, with their individual and combined predictive contribution evaluated later.

- These findings represent associations with fraud and should not be interpreted as causal relationships.

In [36]:
device_counts = train_identity["DeviceInfo"].value_counts(dropna=False)

print("=== DeviceInfo Summary ===")
print(f"Total unique values: {train_identity['DeviceInfo'].nunique(dropna=True):,}")
print(f"Missing values: {train_identity['DeviceInfo'].isna().sum():,}")
print(f"Non-missing values: {train_identity['DeviceInfo'].notna().sum():,}")

print("\n=== Most Common DeviceInfo Values ===")
print(device_counts.head(15))

print("\n=== Frequency Distribution ===")
print(f"Values appearing only once: {(device_counts == 1).sum():,}")
print(f"Values appearing <= 5 times: {(device_counts <= 5).sum():,}")
print(f"Values appearing > 100 times: {(device_counts > 100).sum():,}")

=== DeviceInfo Summary ===
Total unique values: 1,786
Missing values: 25,567
Non-missing values: 118,666

=== Most Common DeviceInfo Values ===
DeviceInfo
Windows                        47722
NaN                            25567
iOS Device                     19782
MacOS                          12573
Trident/7.0                     7440
rv:11.0                         1901
rv:57.0                          962
SM-J700M Build/MMB29K            549
SM-G610M Build/MMB29K            461
SM-G531H Build/LMY48B            410
rv:59.0                          362
SM-G935F Build/NRD90M            334
SM-G955U Build/NRD90M            328
SM-G532M Build/MMB29T            316
ALE-L23 Build/HuaweiALE-L23      312
Name: count, dtype: int64

=== Frequency Distribution ===
Values appearing only once: 440
Values appearing <= 5 times: 1,018
Values appearing > 100 times: 64


### DeviceInfo Observations

- `DeviceInfo` contains **1,786 unique non-missing values** across 118,666 non-missing records.

- The feature is highly fragmented: **1,018 unique values occur five times or fewer**, while only 64 values occur more than 100 times.

- Several common values such as `Windows`, `iOS Device`, and `MacOS` provide relatively broad device information, while other values contain more specific device or browser strings.

- Directly one-hot encoding all raw `DeviceInfo` categories could create a high-dimensional representation containing many extremely rare categories.

- `DeviceInfo` will therefore be treated as a **high-cardinality categorical feature**.

- Its representation will be evaluated using approaches such as rare-category grouping, frequency-based features, or carefully selected higher-level characteristics.

- Any frequency-based feature will be calculated using information available before the transaction being scored to avoid data leakage.

In [38]:
id_cols = [col for col in train_identity.columns if col.startswith("id_")]

print("Number of id_* features:", len(id_cols))

print("\n=== Missing Values ===")
print(
    train_identity[id_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n=== Unique Values ===")
print(
    train_identity[id_cols]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

Number of id_* features: 38

=== Missing Values ===
id_24    139486
id_25    139101
id_07    139078
id_08    139078
id_21    139074
id_26    139070
id_22    139064
id_27    139064
id_23    139064
id_18     99120
id_04     77909
id_03     77909
id_33     70944
id_10     69307
id_09     69307
id_30     66668
id_32     66647
id_34     66428
id_14     64189
id_13     16913
id_16     14893
id_06      7368
id_05      7368
id_20      4972
id_19      4915
id_17      4864
id_31      3951
id_02      3361
id_28      3255
id_29      3255
id_11      3255
id_35      3248
id_36      3248
id_37      3248
id_38      3248
id_15      3248
id_12         0
id_01         0
dtype: int64

=== Unique Values ===
id_02    115655
id_19       522
id_21       490
id_20       394
id_11       365
id_25       341
id_33       260
id_31       130
id_17       104
id_06       101
id_26        95
id_08        94
id_05        93
id_07        84
id_01        77
id_30        75
id_10        62
id_13        54
id_09        46


## Anonymized Identity Feature Observations

- The identity dataset contains **38 anonymized `id_*` features** with highly variable levels of missingness and cardinality.

- Several features have extremely high missingness, with `id_24` missing for approximately **96.7%** of identity records.

- In contrast, features such as `id_02` have relatively high availability and very high cardinality, with more than **115,000 unique non-missing values**.

- High missingness will not be used as an automatic exclusion criterion because **missingness itself may contain predictive information in fraud detection**.

- High-cardinality anonymized features will also not be blindly one-hot encoded because this could create a sparse and inefficient representation.

- Since the business meaning of the anonymized `id_*` variables is not reliably known, they will be evaluated based on **missingness, predictive contribution, leakage risk, temporal stability, and redundancy**.

In [40]:
identity_missing = train_identity[["TransactionID"] + id_cols].copy()

for col in id_cols:
    identity_missing[col] = identity_missing[col].isna()

identity_missing.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38
0,2987004,False,False,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
1,2987008,False,False,True,True,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False
2,2987010,False,False,False,False,False,False,True,True,False,...,False,True,False,True,True,True,False,False,False,False
3,2987011,False,False,True,True,False,False,True,True,True,...,False,True,False,True,True,True,False,False,False,False
4,2987016,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False


In [43]:
print("=== Missingness Analysis for Identity Features ===\n")

for col in id_cols:
    missing_count = id_missing_fraud[col].sum()
    available_count = (~id_missing_fraud[col]).sum()

    missing_fraud_rate = id_missing_fraud.loc[
        id_missing_fraud[col] == True, "isFraud"
    ].mean()

    available_fraud_rate = id_missing_fraud.loc[
        id_missing_fraud[col] == False, "isFraud"
    ].mean()

    print(f"{col}")
    print(f"  Missing observations   : {missing_count:,}")
    print(f"  Available observations : {available_count:,}")
    print(f"  Missing fraud rate     : {missing_fraud_rate * 100:.2f}%")
    print(f"  Available fraud rate   : {available_fraud_rate * 100:.2f}%")
    print("-" * 50)

=== Missingness Analysis for Identity Features ===

id_01
  Missing observations   : 0
  Available observations : 144,233
  Missing fraud rate     : nan%
  Available fraud rate   : 7.85%
--------------------------------------------------
id_02
  Missing observations   : 3,361
  Available observations : 140,872
  Missing fraud rate     : 2.86%
  Available fraud rate   : 7.97%
--------------------------------------------------
id_03
  Missing observations   : 77,909
  Available observations : 66,324
  Missing fraud rate     : 5.40%
  Available fraud rate   : 10.72%
--------------------------------------------------
id_04
  Missing observations   : 77,909
  Available observations : 66,324
  Missing fraud rate     : 5.40%
  Available fraud rate   : 10.72%
--------------------------------------------------
id_05
  Missing observations   : 7,368
  Available observations : 136,865
  Missing fraud rate     : 4.56%
  Available fraud rate   : 8.02%
-----------------------------------------------

In [45]:
# Analyze whether the fraud rate changes over time

print("=== Fraud Rate Over Time ===\n")

# Divide transactions into 10 chronological groups
# Each group contains roughly the same number of transactions
train_transaction["time_period"] = pd.qcut(
    train_transaction["TransactionDT"],
    q=10,
    duplicates="drop"
)

# Calculate transaction count, fraud count, and fraud rate for each period
time_fraud = train_transaction.groupby(
    "time_period",
    observed=True
)["isFraud"].agg(
    transaction_count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

# Print the results clearly for each time period
for period, row in time_fraud.iterrows():
    print(f"Period: {period}")
    print(f"  Transactions : {int(row['transaction_count']):,}")
    print(f"  Fraud cases  : {int(row['fraud_count']):,}")
    print(f"  Fraud rate   : {row['fraud_rate'] * 100:.2f}%")
    print("-" * 50)

=== Fraud Rate Over Time ===

Period: (86399.999, 1361004.4]
  Transactions : 59,054
  Fraud cases  : 1,631
  Fraud rate   : 2.76%
--------------------------------------------------
Period: (1361004.4, 2310159.6]
  Transactions : 59,054
  Fraud cases  : 1,195
  Fraud rate   : 2.02%
--------------------------------------------------
Period: (2310159.6, 3864163.9]
  Transactions : 59,054
  Fraud cases  : 2,200
  Fraud rate   : 3.73%
--------------------------------------------------
Period: (3864163.9, 5592303.6]
  Transactions : 59,054
  Fraud cases  : 2,532
  Fraud rate   : 4.29%
--------------------------------------------------
Period: (5592303.6, 7306527.5]
  Transactions : 59,054
  Fraud cases  : 2,336
  Fraud rate   : 3.96%
--------------------------------------------------
Period: (7306527.5, 8745782.4]
  Transactions : 59,054
  Fraud cases  : 2,094
  Fraud rate   : 3.55%
--------------------------------------------------
Period: (8745782.4, 10437998.1]
  Transactions : 59,054
  

In [46]:
# Analyze how the fraud rate changes over the timeline of the dataset

print("=== Fraud Rate Over Time ===\n")

# Convert TransactionDT from seconds into approximate days
train_transaction["time_days"] = train_transaction["TransactionDT"] / (24 * 60 * 60)

# Divide transactions into 10 chronological groups
# Each group contains roughly the same number of transactions
train_transaction["time_period"] = pd.qcut(
    train_transaction["TransactionDT"],
    q=10,
    duplicates="drop"
)

# Calculate transaction count, fraud count, and fraud rate for each period
time_fraud = train_transaction.groupby(
    "time_period",
    observed=True
)["isFraud"].agg(
    transaction_count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

# Print the results with approximate elapsed days
for period, row in time_fraud.iterrows():

    # Get the start and end TransactionDT values for this period
    start_dt = period.left
    end_dt = period.right

    # Convert seconds into approximate days
    start_days = start_dt / (24 * 60 * 60)
    end_days = end_dt / (24 * 60 * 60)

    print(f"Period: Day {start_days:.1f} → Day {end_days:.1f}")
    print(f"  Transactions : {int(row['transaction_count']):,}")
    print(f"  Fraud cases  : {int(row['fraud_count']):,}")
    print(f"  Fraud rate   : {row['fraud_rate'] * 100:.2f}%")
    print("-" * 55)

=== Fraud Rate Over Time ===

Period: Day 1.0 → Day 15.8
  Transactions : 59,054
  Fraud cases  : 1,631
  Fraud rate   : 2.76%
-------------------------------------------------------
Period: Day 15.8 → Day 26.7
  Transactions : 59,054
  Fraud cases  : 1,195
  Fraud rate   : 2.02%
-------------------------------------------------------
Period: Day 26.7 → Day 44.7
  Transactions : 59,054
  Fraud cases  : 2,200
  Fraud rate   : 3.73%
-------------------------------------------------------
Period: Day 44.7 → Day 64.7
  Transactions : 59,054
  Fraud cases  : 2,532
  Fraud rate   : 4.29%
-------------------------------------------------------
Period: Day 64.7 → Day 84.6
  Transactions : 59,054
  Fraud cases  : 2,336
  Fraud rate   : 3.96%
-------------------------------------------------------
Period: Day 84.6 → Day 101.2
  Transactions : 59,054
  Fraud cases  : 2,094
  Fraud rate   : 3.55%
-------------------------------------------------------
Period: Day 101.2 → Day 120.8
  Transactions :

## Temporal Fraud Patterns

`TransactionDT` represents the elapsed time since the dataset's reference point. 
Since fraud behavior can change over time, we examine whether the fraud rate remains
stable throughout the observation period.

The analysis will help determine whether a **time-aware validation strategy** is
appropriate for the modeling stage.

In [47]:
# Analyze how fraud rate changes across the transaction timeline

print("=== Fraud Rate Over Time ===\n")

# Divide transactions into 10 chronological groups
# Each group contains roughly the same number of transactions
train_transaction["time_period"] = pd.qcut(
    train_transaction["TransactionDT"],
    q=10,
    duplicates="drop"
)

# Calculate transaction count, fraud count, and fraud rate
time_fraud = train_transaction.groupby(
    "time_period",
    observed=True
)["isFraud"].agg(
    transaction_count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

# Display each period using days and hours
for period, row in time_fraud.iterrows():

    start_dt = period.left
    end_dt = period.right

    # Convert seconds into days and hours
    start_days = int(start_dt // 86400)
    start_hours = int((start_dt % 86400) // 3600)

    end_days = int(end_dt // 86400)
    end_hours = int((end_dt % 86400) // 3600)

    print(
        f"Period: Day {start_days}, Hour {start_hours} "
        f"→ Day {end_days}, Hour {end_hours}"
    )
    print(f"  Transactions : {int(row['transaction_count']):,}")
    print(f"  Fraud cases  : {int(row['fraud_count']):,}")
    print(f"  Fraud rate   : {row['fraud_rate'] * 100:.2f}%")
    print("-" * 55)

=== Fraud Rate Over Time ===

Period: Day 0, Hour 23 → Day 15, Hour 18
  Transactions : 59,054
  Fraud cases  : 1,631
  Fraud rate   : 2.76%
-------------------------------------------------------
Period: Day 15, Hour 18 → Day 26, Hour 17
  Transactions : 59,054
  Fraud cases  : 1,195
  Fraud rate   : 2.02%
-------------------------------------------------------
Period: Day 26, Hour 17 → Day 44, Hour 17
  Transactions : 59,054
  Fraud cases  : 2,200
  Fraud rate   : 3.73%
-------------------------------------------------------
Period: Day 44, Hour 17 → Day 64, Hour 17
  Transactions : 59,054
  Fraud cases  : 2,532
  Fraud rate   : 4.29%
-------------------------------------------------------
Period: Day 64, Hour 17 → Day 84, Hour 13
  Transactions : 59,054
  Fraud cases  : 2,336
  Fraud rate   : 3.96%
-------------------------------------------------------
Period: Day 84, Hour 13 → Day 101, Hour 5
  Transactions : 59,054
  Fraud cases  : 2,094
  Fraud rate   : 3.55%
-------------------

## Leakage Investigation

A fraud model must use only information that would be available at the
**transaction prediction point**.

Features containing information generated after the transaction, directly
encoding the target, or derived from future events can cause **data leakage**.

We will therefore screen the available variables for potential leakage before
feature engineering and model training.

In [48]:
# Look for columns that may need special attention for leakage

print("Target column:")
print("isFraud")

print("\nIdentifier columns:")
print([
    col for col in train_transaction.columns
    if "ID" in col
])

print("\nColumns containing 'fraud':")
print([
    col for col in train_transaction.columns
    if "fraud" in col.lower()
])

Target column:
isFraud

Identifier columns:
['TransactionID']

Columns containing 'fraud':
['isFraud']


In [49]:
# Check the D-features

d_cols = [col for col in train_transaction.columns if col.startswith("D")]

print("Number of D-features:", len(d_cols))

print("\nD-feature summary:")
print(train_transaction[d_cols].describe().T.head(15))

Number of D-features: 15

D-feature summary:
        count        mean         std    min        25%        50%  \
D1   589271.0   94.347568  157.660387    0.0   0.000000   3.000000   
D2   309743.0  169.563231  177.315865    0.0  26.000000  97.000000   
D3   327662.0   28.343348   62.384721    0.0   1.000000   8.000000   
D4   421618.0  140.002441  191.096774 -122.0   0.000000  26.000000   
D5   280699.0   42.335965   89.000144    0.0   1.000000  10.000000   
D6    73187.0   69.805717  143.669253  -83.0   0.000000   0.000000   
D7    38917.0   41.638950   99.743264    0.0   0.000000   0.000000   
D8    74926.0  146.058108  231.663840    0.0   0.958333  37.875000   
D9    74926.0    0.561057    0.316880    0.0   0.208333   0.666666   
D10  514518.0  123.982137  182.615225    0.0   0.000000  15.000000   
D11  311253.0  146.621465  186.042622  -53.0   0.000000  43.000000   
D12   64717.0   54.037533  124.274558  -83.0   0.000000   0.000000   
D13   61952.0   17.901295   67.614425    0.0 

In [50]:
# Check relationship between D-features and transaction time

d_time_corr = train_transaction[d_cols + ["TransactionDT"]].corr()["TransactionDT"]

print(
    d_time_corr
    .drop("TransactionDT")
    .abs()
    .sort_values(ascending=False)
)

D11    0.101266
D14    0.097509
D6     0.084581
D1     0.074031
D15    0.072791
D7     0.070221
D8     0.068752
D4     0.059797
D10    0.058409
D12    0.053108
D2     0.027109
D13    0.024405
D9     0.013735
D3     0.007200
D5     0.001767
Name: TransactionDT, dtype: float64


### D-Feature Leakage Screening

- The D-features were screened for their relationship with `TransactionDT`, the primary transaction-time variable.

- The correlations were relatively low, with the highest absolute correlation being approximately **0.10**.

- Therefore, there is no strong evidence that the D-features are simply encoding transaction time.

- The D-features will be retained as **candidate modeling features** rather than removed solely because they are anonymized.

- Final feature selection will additionally consider predictive performance, temporal stability, missingness, redundancy, and availability at the prediction point.

In [51]:
print("Rows:", len(train_transaction))
print("Unique transactions:", train_transaction["TransactionID"].nunique())

print("\nTarget values:")
print(train_transaction["isFraud"].unique())

print("\nTarget missing values:")
print(train_transaction["isFraud"].isna().sum())

Rows: 590540
Unique transactions: 590540

Target values:
[0 1]

Target missing values:
0


## Final Prediction Definition

- **Unit of prediction:** One transaction
- **Target:** `isFraud`
- `0` = legitimate transaction
- `1` = fraudulent transaction
- **Prediction point:** Information available at transaction time
- `TransactionID` will be retained for identification and joins but excluded from modeling.
- Features identified as leakage or unavailable at prediction time will be excluded.
- Validation will follow a **chronological train → future validation** strategy.
- The final objective is to produce a **fraud probability/risk score** that can support investigation prioritization.

## Phase 1 Complete ✅
### We've now established:
Dataset → structure → target → imbalance → missingness → feature families → identity relationship → device patterns → anonymized features → temporal behavior → leakage → prediction definition

In [52]:
# Merge transaction and identity data

df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Analytical dataset shape:", df.shape)
print("Unique transactions:", df["TransactionID"].nunique())

Analytical dataset shape: (590540, 437)
Unique transactions: 590540


## Analytical Dataset

- The transaction and identity tables were merged using `TransactionID` with a left join.

- The resulting analytical dataset contains **590,540 transactions and 437 columns**.

- All transactions were preserved and `TransactionID` remains unique after the merge.

- Transactions without identity information retain missing values for the corresponding identity features.

- This merged dataset will serve as the starting point for the EDA and feature-engineering pipeline.

In [53]:
df.to_csv("../data/processed/analytical_dataset.csv", index=False)